# Ollama 多模型存储原理

> **本节回答两个问题：**
>
> 1. Ollama 可以本地切换不同模型，那 `ollama_models/` 在装了多个模型后会变成什么样？
> 2. 是「每个模型一个文件夹」？还是「所有权重平铺到 `blobs/`，`manifests/` 只存目录指针」？

---

## 1. 直接给答案

**答案是第二种**：所有数据（模型权重、tokenizer、template、license 等）都以 sha256 文件名**平铺**在 `blobs/` 目录下；`manifests/` 是一棵按「**registry / 命名空间 / 模型名 / 标签**」组织的目录树，每个叶节点是一份 JSON，指向若干 blob。

下面这张图把关系一次说清：

```text
ollama_models/
├── blobs/                                  ← 唯一的数据池子（CAS）
│   ├── sha256-aaa...   (4.4 GB)            ← 某个模型的权重
│   ├── sha256-bbb...   (148 B)             ← 某份 params 配置
│   ├── sha256-ccc...   (556 B)             ← 某份对话模板
│   └── ...
│
└── manifests/                              ← 按"人话名字"组织的目录树
    └── registry.ollama.ai/
        └── library/
            ├── deepseek-r1/
            │   └── 7b                      ← JSON 文件，指向 blobs/sha256-aaa, ...
            ├── qwen3/
            │   └── 8b                      ← JSON 文件，可能也指向 blobs/sha256-ccc
            └── llama3.2/
                └── 3b                      ← JSON 文件，可能指向 blobs/sha256-bbb
```

**核心要点**：

- `blobs/` 永远是一个**扁平大池子**，里面的文件名就是其内容的 sha256
- `manifests/` 才是按"模型名:标签"组织的目录树
- 多个模型可以**指向同一个 blob**（去重）

## 2. 错误直觉 vs Ollama 真实做法

很多人第一反应（包括我直到深挖前）都会以为是"每个模型一个文件夹"，因为大多数模型仓库（如 Hugging Face 早期格式、PyTorch checkpoint）就是这么放的：

### ❌ 错误直觉

```text
ollama_models/
├── deepseek-r1-7b/
│   ├── model.gguf
│   ├── tokenizer.json
│   ├── template.txt
│   └── license.txt
├── qwen3-8b/
│   ├── model.gguf
│   ├── tokenizer.json
│   ├── template.txt
│   └── license.txt
└── llama3.2-3b/
    └── ...
```

如果是这样，**两个模型就算 tokenizer 内容完全相同，也要存两份**，浪费空间。

### ✅ Ollama 实际做法（Docker / CAS 同款）

```text
ollama_models/
├── blobs/
│   ├── sha256-aaa... (4.4G) ← deepseek 权重
│   ├── sha256-ddd... (5.0G) ← qwen 权重
│   ├── sha256-eee... (2.0G) ← llama 权重
│   ├── sha256-ccc... (556B) ← deepseek 模板
│   ├── sha256-fff... (612B) ← qwen 模板
│   └── sha256-bbb... (148B) ← 通用 stop tokens 配置  
│                            (如果两个模型 stop tokens 一样，
│                            就只存这一份，被多个 manifest 共同引用)
│
└── manifests/registry.ollama.ai/library/
    ├── deepseek-r1/7b  → 指向 aaa, ccc, bbb
    ├── qwen3/8b        → 指向 ddd, fff, bbb  ← 共享 bbb，不重复存
    └── llama3.2/3b     → 指向 eee, ccc, bbb  ← 共享 ccc 和 bbb
```

**节省空间的关键**：内容相同的小 blob（如 stop tokens 配置、license 文件）会被自动复用，不会重复占磁盘。

---

## 3. 为什么这么设计？

回顾一下 [01笔记](./01笔记.ipynb) 之外补充的 Ollama 设计哲学：

| 设计目标 | 平铺 CAS 怎么解决 |
|---|---|
| **去重节省空间** | 同内容 → 同 sha256 → 同一份文件 |
| **完整性校验** | 每次加载时可重算 sha256 比对文件名 |
| **断点续传** | 一个大 blob 分片下载，按 sha256 校验后写入 |
| **垃圾回收** | 扫所有 manifest，没人引用的 blob 安全删 |
| **删模型不删别人的** | 只删 manifest，再 GC，自动只清理"孤儿 blob" |

## 4. 假设：我们现在再装 3 个模型，目录会怎么变？

当前状态（只装了 `deepseek-r1:7b`）：

```text
ollama_models/
├── blobs/   (5 个文件，4.4 GB)
└── manifests/registry.ollama.ai/library/deepseek-r1/7b
```

假设接下来运行：

```bash
ollama pull qwen3:8b
ollama pull llama3.2:3b
ollama pull gemma2:2b
```

**变化的部分**：

```text
ollama_models/
├── blobs/                                ← 文件数会增加，但远小于 4 × 5 = 20 个
│   ├── sha256-...                        ← deepseek 原有 5 个
│   ├── sha256-...   (新 qwen3 权重 ~5 GB)
│   ├── sha256-...   (新 qwen3 模板)
│   ├── sha256-...   (新 llama3.2 权重 ~2 GB)
│   ├── sha256-...   (新 gemma2 权重 ~1.6 GB)
│   └── ... (可能少量小 blob 被共享，不再重复)
│
└── manifests/registry.ollama.ai/library/
    ├── deepseek-r1/7b
    ├── qwen3/8b        ← 新增
    ├── llama3.2/3b     ← 新增
    └── gemma2/2b       ← 新增
```

**重点观察**：

1. **`manifests/` 像一棵树长出新枝**：每个新模型在 `library/<模型名>/<标签>` 多出一个 JSON 文件
2. **`blobs/` 只是一堆新文件加进来**，**不会分子目录**
3. **同标签同名（如 `:7b`）多次 pull 不会重复**：sha256 一样直接复用

---

## 5. 自己上手验证（命令演练）

不实际拉模型也能演练命令，等 Day 3、Day 4 真要做多模型对比时再实操：

```bash
# 列出所有本地模型（manifests 的人话名字）
ollama list

# 看某个模型的"组成清单"（manifest）
ollama show deepseek-r1:7b
ollama show deepseek-r1:7b --modelfile      # 看 Modelfile（template + params 等）
ollama show deepseek-r1:7b --parameters     # 看推理默认参数
ollama show deepseek-r1:7b --template       # 看对话模板

# 拉一个新模型（自动复用已有 blob）
ollama pull qwen3:8b

# 删除一个模型（只删 manifest 引用，再 GC 真正没人用的 blob）
ollama rm qwen3:8b

# 看磁盘占用（CAS 平铺下，blobs/ 是大头）
du -sh ollama_models/blobs ollama_models/manifests
```

---

## 6. 备份策略

**整个 `ollama_models/` 文件夹一起备份就够了**，因为：

- `manifests/` 描述了"什么名字对应什么 blob"
- `blobs/` 是实际数据
- 两者一起拷过去，任何机器上启动 Ollama 都能立刻识别

**反过来：千万别只拷 `blobs/` 不拷 `manifests/`**，因为没有 manifest，blob 就是"一堆没有名字的二进制"，Ollama 不知道它们组成了哪个模型。

---

## 7. 与 Docker 的横向类比

| 概念 | Docker | Ollama |
|---|---|---|
| 内容池 | `/var/lib/docker/overlay2/<hash>/` | `ollama_models/blobs/sha256-<hash>` |
| 名字索引 | `nginx:latest` → image manifest | `deepseek-r1:7b` → manifest 文件 |
| 命令 `pull` | `docker pull nginx:latest` | `ollama pull deepseek-r1:7b` |
| 命令 `list` | `docker images` | `ollama list` |
| 命令 `rm` | `docker rmi nginx:latest` | `ollama rm deepseek-r1:7b` |
| 自动去重 | 多个镜像共享 layer | 多个模型共享 blob |

> **底层是同一种思想**：Content-Addressable Storage（CAS）。

---

## 8. 给 RAG 多模型对比的启示

接下来 [02 LangChain_RAG](../02%20LangChain_RAG/) 要做多模型评测，我们会用 Ollama 同时托管多个本地模型（比如 deepseek-r1:7b、qwen3:8b、llama3.2:3b）。由于 Ollama 的 CAS 设计：

- **不用担心总占用爆炸**：相同家族的小配置 blob 会共享
- **切换模型零成本**：`ollama run <model_name>` 即可，不需要重启服务
- **API 维度统一**：都是 `POST http://127.0.0.1:11434/api/generate`，只是 body 里 `model` 字段变化
- **方便对比基准**：参考 `benchmark_batch.py` 的 `--models qwen3:8b deepseek-r1:7b ...` 思路，统一接口跑测试

> **一句话总结**：Ollama 把模型当作 Docker 镜像管理；多模型实质是「一个 blob 池 + 一棵 manifest 树」，**多模型几乎没有额外组织成本**。